# Estatística descritiva com a PNAD 2015
Tipos de variáveis, distribuições de frequência, medidas de tendência central, medidas separatrizes e medidas de dispersão, aplicadas aos dados da Pesquisa Nacional por Amostra de Domicílios (PNAD) 2015. Termina com um projeto de análise descritiva do mesmo conjunto de dados.

**Origem:** Alura, Curso de Estatística (parte 1): frequências e medidas, com o trabalho de análise descritiva do curso.

**Dados:** `dados.csv` (PNAD 2015 tratada pelo curso). Envie o arquivo para `/content/` no Colab antes de executar.

**Índice**
1. Conhecendo os dados
2. Distribuições de frequências
3. Medidas de tendência central
4. Medidas separatrizes
5. Medidas de dispersão
6. Projeto: análise descritiva

In [ ]:
# Setup: todas as bibliotecas já vêm instaladas no Colab
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

## 1. Conhecendo os dados
Apresentação do dataset do projeto e classificação das variáveis em qualitativas (nominais/ordinais) e quantitativas (discretas/contínuas).

### 1.1 Dataset do projeto
A **PNAD** investiga anualmente características gerais da população, de educação, trabalho, rendimento e habitação, entre outras, e é um importante instrumento para formulação, validação e avaliação de políticas de desenvolvimento socioeconômico no Brasil.

**Fonte dos dados:** https://ww2.ibge.gov.br/home/estatistica/populacao/trabalhoerendimento/pnad2015/microdados.shtm

**Variáveis utilizadas**
- **Renda**: rendimento mensal do trabalho principal para pessoas de 10 anos ou mais de idade.
- **Idade**: idade do morador na data de referência, em anos.
- **Altura** (elaboração própria): altura do morador em metros.
- **UF**: código da unidade da federação (11 Rondônia ... 53 Distrito Federal; dicionário completo na seção 6).
- **Sexo**: 0 Masculino, 1 Feminino.
- **Anos de Estudo**: 1 Sem instrução e menos de 1 ano, 2 = 1 ano, ..., 16 = 15 anos ou mais, 17 Não determinados.
- **Cor**: 0 Indígena, 2 Branca, 4 Preta, 6 Amarela, 8 Parda, 9 Sem declaração.

> **Observação:** nos dados originais foram eliminados os registros com Renda inválida (999 999 999 999) ou ausente, e foram mantidas apenas as **pessoas de referência** de cada domicílio.

In [ ]:
dados = pd.read_csv('dados.csv')
dados.head()

### 1.2 Tipos de variáveis
<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img001.png' width='70%'>

**Qualitativas ordinais:** podem ser ordenadas ou hierarquizadas.

In [ ]:
sorted(dados['Anos de Estudo'].unique())

**Qualitativas nominais:** não podem ser ordenadas ou hierarquizadas.

In [ ]:
for coluna in ['UF', 'Sexo', 'Cor']:
    print(coluna, sorted(dados[coluna].unique()))

**Quantitativas discretas:** representam uma contagem; os valores possíveis formam um conjunto finito ou enumerável.

> A idade pode ser classificada de três formas: **quantitativa discreta** (anos completos), **quantitativa contínua** (idade exata, com frações de ano) ou **qualitativa ordinal** (faixas de idade).

In [ ]:
print('De %s até %s anos' % (dados.Idade.min(), dados.Idade.max()))

**Quantitativas contínuas:** contagem ou mensuração que assume valores em uma escala contínua (números reais).

In [ ]:
print('De %s até %s metros' % (dados.Altura.min(), dados.Altura.max()))

## 2. Distribuições de frequências
O primeiro passo de uma análise é conhecer o comportamento das variáveis. As **distribuições de frequências** e os **histogramas** mostram como os fenômenos em estudo se distribuem.

### 2.1 Variáveis qualitativas
**Método 1:** `value_counts()`.

In [ ]:
frequencia = dados['Sexo'].value_counts()
percentual = dados['Sexo'].value_counts(normalize=True) * 100

dist_freq_qualitativas = pd.DataFrame({'Frequência': frequencia, 'Porcentagem (%)': percentual})
dist_freq_qualitativas.rename(index={0: 'Masculino', 1: 'Feminino'}, inplace=True)
dist_freq_qualitativas.rename_axis('Sexo', axis='columns', inplace=True)
dist_freq_qualitativas

**Método 2:** `pd.crosstab()`, que cruza duas variáveis.

In [ ]:
sexo = {0: 'Masculino',
        1: 'Feminino'}

cor = {0: 'Indígena',
       2: 'Branca',
       4: 'Preta',
       6: 'Amarela',
       8: 'Parda',
       9: 'Sem declaração'}

In [ ]:
frequencia = pd.crosstab(dados.Sexo, dados.Cor)
frequencia.rename(index=sexo, columns=cor, inplace=True)
frequencia

In [ ]:
percentual = pd.crosstab(dados.Sexo, dados.Cor, normalize=True) * 100
percentual.rename(index=sexo, columns=cor, inplace=True)
percentual

Com `aggfunc` e `values`, o crosstab resume uma terceira variável. Aqui, a renda média por sexo e cor:

In [ ]:
renda_media = pd.crosstab(dados.Sexo, dados.Cor, aggfunc='mean', values=dados.Renda)
renda_media.rename(index=sexo, columns=cor, inplace=True)
renda_media

### 2.2 Variáveis quantitativas (classes personalizadas)
**Passo 1 - Especificar os limites de cada classe**, em salários mínimos (SM = R$ 788,00 em 2015):

| Classe | Faixa em SM | Faixa em R$ |
|---|---|---|
| A | Acima de 20 SM | Acima de 15.760 |
| B | De 10 a 20 SM | De 7.880 a 15.760 |
| C | De 4 a 10 SM | De 3.152 a 7.880 |
| D | De 2 a 4 SM | De 1.576 a 3.152 |
| E | Até 2 SM | Até 1.576 |

In [ ]:
print('Renda de %s até %s' % (dados.Renda.min(), dados.Renda.max()))

classes = [0, 1576, 3152, 7880, 15760, 200000]
labels = ['E', 'D', 'C', 'B', 'A']

**Passo 2 - Criar a tabela de frequências** com `pd.cut()`.

In [ ]:
renda_por_classe = pd.cut(x=dados.Renda, bins=classes, labels=labels, include_lowest=True)

frequencia = renda_por_classe.value_counts()
percentual = renda_por_classe.value_counts(normalize=True) * 100

dist_freq_quantitativas_personalizadas = pd.DataFrame(
    {'Frequência': frequencia, 'Porcentagem (%)': percentual}
)
dist_freq_quantitativas_personalizadas.sort_index(ascending=False)

O mesmo procedimento para uma variável contínua, a altura, em três classes:

In [ ]:
classes_altura = [dados.Altura.min(), 1.65, 1.75, dados.Altura.max()]
labels_altura = ['1 - baixa', '2 - média', '3 - alta']

altura_por_classe = pd.cut(x=dados.Altura, bins=classes_altura, labels=labels_altura, include_lowest=True)

dist_freq_altura = pd.DataFrame({
    'Frequência': altura_por_classe.value_counts(),
    'Porcentagem (%)': altura_por_classe.value_counts(normalize=True) * 100
})
dist_freq_altura.rename_axis('Estaturas', axis='columns', inplace=True)
dist_freq_altura.sort_index()

As classes também podem ser cruzadas com outra variável. Renda média por sexo em quatro níveis de renda:

In [ ]:
cortes = [dados.Renda.min(), 2000, 5000, 50000, dados.Renda.max()]
rotulos = ['menor', 'baixo', 'médio', 'alto']
nivel_renda = pd.cut(x=dados.Renda, bins=cortes, labels=rotulos, include_lowest=True)

renda_nivel_sexo = pd.crosstab(dados.Sexo, nivel_renda, aggfunc='mean', values=dados.Renda)
renda_nivel_sexo.rename(index=sexo, inplace=True)
renda_nivel_sexo

### 2.3 Variáveis quantitativas (classes de amplitude fixa)
**Passo 1 - Definir o número de classes** pela regra de Sturges:

$$k = 1 + \frac {10}{3}\log_{10}n$$

In [ ]:
n = dados.shape[0]
k = 1 + (10 / 3) * np.log10(n)
k = int(k.round(0))
k

**Passo 2 - Criar a tabela de frequências** com `k` classes de mesma amplitude.

In [ ]:
renda_amplitude_fixa = pd.cut(x=dados.Renda, bins=k, include_lowest=True)

dist_freq_quantitativas_amplitude_fixa = pd.DataFrame({
    'Frequência': renda_amplitude_fixa.value_counts(sort=False),
    'Porcentagem (%)': renda_amplitude_fixa.value_counts(sort=False, normalize=True) * 100
})
dist_freq_quantitativas_amplitude_fixa

### 2.4 Histograma
O **histograma** é a representação gráfica de uma distribuição de frequências: retângulos lado a lado, cuja área é proporcional à frequência da classe que representam.

In [ ]:
ax = sns.histplot(dados.Altura)
ax.figure.set_size_inches(12, 6)
ax.set_title('Distribuição de Frequências - Altura', fontsize=18)
ax.set_xlabel('Metros', fontsize=14);

In [ ]:
ax = sns.histplot(dados.Altura, kde=True, stat='density')
ax.figure.set_size_inches(12, 6)
ax.set_title('Distribuição de Frequências - Altura - KDE', fontsize=18)
ax.set_xlabel('Metros', fontsize=14);

In [ ]:
dados.Altura.hist(bins=50, figsize=(12, 6));

Gráfico de barras da distribuição por classes de renda (seção 2.2):

In [ ]:
dist_freq_quantitativas_personalizadas['Frequência'].sort_index().plot.bar(width=1, color='blue', alpha=0.2, figsize=(12, 6));

## 3. Medidas de tendência central
Média, mediana e moda, primeiro em um DataFrame pequeno de exemplo (notas de três alunos) e depois no dataset.

In [ ]:
df = pd.DataFrame(data={'Fulano': [8, 10, 4, 8, 6, 10, 8],
                        'Beltrano': [10, 2, 0.5, 1, 3, 9.5, 10],
                        'Sicrano': [7.5, 8, 7, 8, 8, 8.5, 7]},
                  index=['Matemática',
                         'Português',
                         'Inglês',
                         'Geografia',
                         'História',
                         'Física',
                         'Química'])
df.rename_axis('Matérias', axis='columns', inplace=True)
df

### 3.1 Média aritmética
Representada por $\mu$ quando se refere à população e por $\bar{X}$ quando se refere à amostra:

$$\mu = \frac 1n\sum_{i=1}^{n}X_i$$

onde $n$ = número de observações e $X_i$ = valor da i-ésima observação.

In [ ]:
(8 + 10 + 4 + 8 + 6 + 10 + 8) / 7

In [ ]:
df['Fulano'].mean()

In [ ]:
print('Renda média:', dados.Renda.mean())
dados.groupby(['Sexo'])['Renda'].mean().rename(index=sexo)

### 3.2 Mediana
1. Ordenar o conjunto de dados;
2. Identificar o número de observações $n$;
3. Identificar o elemento mediano: $Elemento_{Md} = \frac{n+1}2$ se $n$ for ímpar, ou $Elemento_{Md} = \frac{n}2$ se $n$ for par;
4. Obter a mediana:

$$Md = X_{Elemento_{Md}} \quad (n \text{ ímpar}) \qquad Md = \frac{X_{Elemento_{Md}} + X_{Elemento_{Md}+1}}2 \quad (n \text{ par})$$

**Exemplo 1 - n ímpar**

<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img002.png' width='40%'>

In [ ]:
notas_fulano = df.Fulano.sort_values().reset_index()
notas_fulano

In [ ]:
n = notas_fulano.shape[0]
elemento_md = int((n + 1) / 2)
print('Elemento mediano:', elemento_md)
print('Mediana pela fórmula:', notas_fulano.loc[elemento_md - 1, 'Fulano'])
print('Mediana com median():', notas_fulano['Fulano'].median())

**Exemplo 2 - n par**

<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img003.png' width='50%'>

In [ ]:
notas_beltrano = df.Beltrano.sample(6, random_state=101).sort_values().reset_index()
notas_beltrano

In [ ]:
n = notas_beltrano.shape[0]
elemento_md = int(n / 2)
print('Mediana pela fórmula:', (notas_beltrano.Beltrano[elemento_md - 1] + notas_beltrano.Beltrano[elemento_md]) / 2)
print('Mediana com median():', notas_beltrano['Beltrano'].median())

No dataset, `median()` equivale a `quantile()` com o valor padrão 0.5:

In [ ]:
dados.Renda.median(), dados.Renda.quantile()

### 3.3 Moda
A moda é o valor mais frequente de um conjunto de dados, bastante utilizada para dados qualitativos. Pode haver mais de uma moda.

In [ ]:
df.mode()

In [ ]:
exemplo = pd.Series([1, 2, 2, 3, 4, 4, 5, 6, 7])
exemplo.mode()

In [ ]:
dados.Renda.mode()

In [ ]:
dados.Altura.mode()

### 3.4 Relação entre média, mediana e moda
<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img004.png' width='80%'>

**Renda:** assimétrica à direita (moda < mediana < média).

In [ ]:
ax = sns.histplot(dados.query('Renda < 20000').Renda, kde=True, stat='density')
ax.figure.set_size_inches(12, 6)

In [ ]:
Moda = dados.Renda.mode()[0]
Mediana = dados.Renda.median()
Media = dados.Renda.mean()
print(Moda, Mediana, Media)
Moda < Mediana < Media

**Altura:** distribuição simétrica (média, mediana e moda praticamente iguais; a altura é multimodal).

In [ ]:
ax = sns.histplot(dados.Altura, kde=True, stat='density')
ax.figure.set_size_inches(12, 6)

In [ ]:
dados.Altura.mode(), dados.Altura.median(), dados.Altura.mean()

**Anos de Estudo:** assimétrica à esquerda (moda > mediana > média).

In [ ]:
ax = sns.histplot(dados['Anos de Estudo'], bins=17, kde=True, stat='density')
ax.figure.set_size_inches(12, 6)

In [ ]:
Moda = dados['Anos de Estudo'].mode()[0]
Mediana = dados['Anos de Estudo'].median()
Media = dados['Anos de Estudo'].mean()
print(Moda, Mediana, Media)
Moda > Mediana > Media

## 4. Medidas separatrizes
### 4.1 Quartis, decis e percentis
São medidas de posição semelhantes à mediana, embora não sejam de tendência central: a mediana divide a distribuição em duas partes iguais, os quartis em quatro, os decis em dez e os percentis em cem.

In [ ]:
dados.Renda.quantile([0.25, 0.5, 0.75])

In [ ]:
dados.Renda.quantile([i / 10 for i in range(1, 10)])

In [ ]:
dados.Renda.quantile([i / 100 for i in range(1, 100)])

Distribuição de frequências acumulada da idade e seus decis:

In [ ]:
ax = sns.histplot(dados.Idade, bins=10, cumulative=True, stat='density', kde=True)
ax.figure.set_size_inches(14, 6)
ax.set_title('Distribuição de Frequências Acumulada', fontsize=18)
ax.set_ylabel('Acumulado', fontsize=14)
ax.set_xlabel('Anos', fontsize=14);

In [ ]:
dados.Idade.quantile([i / 10 for i in range(1, 10)])

### 4.2 Box-plot
O box plot dá uma ideia da posição, dispersão, assimetria, caudas e dados discrepantes (outliers). A posição central é dada pela mediana e a dispersão pelo intervalo interquartil ($IIQ = Q3 - Q1$); as posições relativas de $Q1$, mediana e $Q3$ indicam a simetria da distribuição.

<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img005.png' width='65%'>

In [ ]:
ax = sns.boxplot(x='Altura', data=dados, orient='h')
ax.figure.set_size_inches(12, 4)
ax.set_title('Altura', fontsize=18)
ax.set_xlabel('Metros', fontsize=14);

In [ ]:
ax = sns.boxplot(x='Altura', y='Sexo', data=dados, orient='h')
ax.figure.set_size_inches(12, 4)
ax.set_title('Altura por sexo (0 = Masculino, 1 = Feminino)', fontsize=18)
ax.set_xlabel('Metros', fontsize=14);

In [ ]:
ax = sns.boxplot(x='Renda', y='Sexo', data=dados.query('Renda < 10000'), orient='h')
ax.figure.set_size_inches(12, 4)
ax.set_title('Renda por sexo (até R$ 10.000)', fontsize=18)
ax.set_xlabel('R$', fontsize=14);

In [ ]:
ax = sns.boxplot(x='Anos de Estudo', y='Sexo', data=dados, orient='h')
ax.figure.set_size_inches(12, 4)
ax.set_title('Anos de Estudo por sexo', fontsize=18)
ax.set_xlabel('Anos', fontsize=14);

<img src='https://caelum-online-public.s3.amazonaws.com/1177-estatistica-parte1/01/img006.png' width='80%'>

## 5. Medidas de dispersão
As medidas de posição podem não bastar para caracterizar conjuntos distintos, especialmente quando as observações são muito dispersas. Usamos de novo as notas de Fulano.

### 5.1 Desvio médio absoluto
$$DM = \frac 1n\sum_{i=1}^{n}|X_i-\bar{X}|$$

In [ ]:
notas_fulano = df[['Fulano']].copy()
nota_media_fulano = notas_fulano['Fulano'].mean()
notas_fulano['Desvio'] = notas_fulano['Fulano'] - nota_media_fulano
notas_fulano['|Desvio|'] = notas_fulano['Desvio'].abs()
print('Soma dos desvios (sempre zero):', notas_fulano['Desvio'].sum())
notas_fulano

In [ ]:
ax = notas_fulano['Fulano'].plot(style='o')
ax.figure.set_size_inches(14, 6)
ax.hlines(y=nota_media_fulano, xmin=0, xmax=notas_fulano.shape[0] - 1, colors='red')
for i in range(notas_fulano.shape[0]):
    ax.vlines(x=i, ymin=nota_media_fulano, ymax=notas_fulano['Fulano'].iloc[i], linestyles='dashed')

In [ ]:
# Series.mad() foi removido no pandas 2.0: o desvio médio absoluto é a média dos |desvios|
desvio_medio_absoluto = notas_fulano['|Desvio|'].mean()
desvio_medio_absoluto

### 5.2 Variância
Construída a partir dos desvios em torno da média, elevados ao quadrado.

**Variância populacional:** $$\sigma^2 = \frac 1n\sum_{i=1}^{n}(X_i-\mu)^2$$

**Variância amostral:** $$S^2 = \frac 1{n-1}\sum_{i=1}^{n}(X_i-\bar{X})^2$$

In [ ]:
notas_fulano['(Desvio)^2'] = notas_fulano['Desvio'].pow(2)
print('Pela fórmula:', notas_fulano['(Desvio)^2'].sum() / (len(notas_fulano) - 1))
variancia = notas_fulano['Fulano'].var()
variancia

### 5.3 Desvio padrão
A variância é expressa no quadrado da unidade original, o que dificulta a comparação. O desvio padrão é a sua raiz quadrada.

**Populacional:** $$\sigma = \sqrt{\frac 1n\sum_{i=1}^{n}(X_i-\mu)^2} \Longrightarrow \sigma = \sqrt{\sigma^2}$$

**Amostral:** $$S = \sqrt{\frac 1{n-1}\sum_{i=1}^{n}(X_i-\bar{X})^2} \Longrightarrow S = \sqrt{S^2}$$

In [ ]:
print('Raiz da variância:', np.sqrt(variancia))
desvio_padrao = notas_fulano['Fulano'].std()
desvio_padrao

Resumo das medidas para os três alunos: médias e medianas parecidas podem esconder dispersões bem diferentes (Beltrano).

In [ ]:
pd.DataFrame({'Média': df.mean(), 'Mediana': df.median(), 'Desvio padrão': df.std()})

In [ ]:
df.mode()

## 6. Projeto: análise descritiva
Trabalho final do curso: análise descritiva básica da PNAD 2015 com as ferramentas vistas acima (tabelas de frequência, cruzamentos, separatrizes, dispersão e box plots). Os itens que só repetiam cálculos das seções 2 a 5 (média, mediana, moda, desvios e histogramas da renda) foram omitidos.

Dicionários para renomear linhas e colunas (`sexo` e `cor` foram definidos na seção 2.1):

In [ ]:
anos_de_estudo = {
    1: 'Sem instrução e menos de 1 ano', 2: '1 ano', 3: '2 anos', 4: '3 anos', 5: '4 anos',
    6: '5 anos', 7: '6 anos', 8: '7 anos', 9: '8 anos', 10: '9 anos', 11: '10 anos',
    12: '11 anos', 13: '12 anos', 14: '13 anos', 15: '14 anos', 16: '15 anos ou mais',
    17: 'Não determinados'
}
uf = {
    11: 'Rondônia', 12: 'Acre', 13: 'Amazonas', 14: 'Roraima', 15: 'Pará', 16: 'Amapá',
    17: 'Tocantins', 21: 'Maranhão', 22: 'Piauí', 23: 'Ceará', 24: 'Rio Grande do Norte',
    25: 'Paraíba', 26: 'Pernambuco', 27: 'Alagoas', 28: 'Sergipe', 29: 'Bahia',
    31: 'Minas Gerais', 32: 'Espírito Santo', 33: 'Rio de Janeiro', 35: 'São Paulo',
    41: 'Paraná', 42: 'Santa Catarina', 43: 'Rio Grande do Sul', 50: 'Mato Grosso do Sul',
    51: 'Mato Grosso', 52: 'Goiás', 53: 'Distrito Federal'
}

### 6.1 Classes de renda em salários mínimos
**A** acima de 25 SM, **B** de 15 a 25 SM, **C** de 5 a 15 SM, **D** de 2 a 5 SM, **E** até 2 SM, com SM = R$ 788,00.

In [ ]:
sm = 788
renda_classes = [0, 2 * sm, 5 * sm, 15 * sm, 25 * sm, dados.Renda.max()]
classes = ['E', 'D', 'C', 'B', 'A']  # lista (ordenada), não conjunto

renda_sm = pd.cut(x=dados.Renda, bins=renda_classes, labels=classes, include_lowest=True)
indicadores_renda = pd.DataFrame({
    'Frequência': renda_sm.value_counts(),
    'Percentual': renda_sm.value_counts(normalize=True) * 100
}).sort_index()
indicadores_renda

In [ ]:
sns.barplot(x=indicadores_renda.index, y=indicadores_renda['Frequência']);

### 6.2 Renda segundo sexo e cor
Medidas de tendência central e de dispersão em uma única tabulação: `aggfunc` do `crosstab()` aceita uma lista de funções.

In [ ]:
renda_sexo_cor = pd.crosstab(dados.Cor, dados.Sexo, values=dados.Renda, aggfunc=['mean', 'median', 'max'])
renda_sexo_cor.rename(index=cor, columns=sexo, inplace=True)
renda_sexo_cor

In [ ]:
dispersao_sexo_cor = pd.crosstab(dados.Cor, dados.Sexo, values=dados.Renda, aggfunc=['var', 'std'])
dispersao_sexo_cor.rename(index=cor, columns=sexo, inplace=True)
dispersao_sexo_cor

In [ ]:
renda_ate_10k = dados.query('Renda < 10000').assign(Cor=lambda d: d.Cor.map(cor),
                                                    Sexo=lambda d: d.Sexo.map(sexo))
ax = sns.boxplot(x='Renda', y='Cor', hue='Sexo', data=renda_ate_10k, orient='h')
ax.figure.set_size_inches(14, 8)
ax.set_title('Box-plot da renda por cor e sexo (até R$ 10.000)', fontsize=18);

### 6.3 Desafio: separatrizes da renda
Percentual de pessoas que ganham um salário mínimo (R$ 788,00) ou menos, com `percentileofscore()` do scipy, e o valor máximo ganho por 99% das pessoas, com `quantile()`.

In [ ]:
percentual_ate_sm = stats.percentileofscore(dados.Renda, sm, kind='weak')
print(f'{percentual_ate_sm:.2f}% das pessoas ganham até um salário mínimo')

renda_99 = dados.Renda.quantile(0.99)
print(f'99% das pessoas ganham até R$ {renda_99:,.2f}')

### 6.4 Renda segundo anos de estudo e sexo

In [ ]:
renda_estudo_sexo = pd.crosstab(dados['Anos de Estudo'], dados.Sexo, values=dados.Renda,
                                aggfunc=['mean', 'median', 'max', 'std'])
renda_estudo_sexo.rename(index=anos_de_estudo, columns=sexo, inplace=True)
renda_estudo_sexo

Para verificar se a diferença entre os sexos se mantém para pessoas da mesma idade, o box plot usa apenas quem tem 50 anos:

In [ ]:
ax = sns.boxplot(x='Renda', y='Anos de Estudo', hue='Sexo',
                 data=dados.query('Renda < 10000 and Idade == 50'), orient='h')
ax.figure.set_size_inches(14, 8)
ax.set_title('Box-plot da renda por anos de estudo e sexo (50 anos, até R$ 10.000)', fontsize=18);

### 6.5 Renda segundo as unidades da federação

In [ ]:
renda_uf = dados.groupby(['UF']).agg({'Renda': ['mean', 'median', 'max', 'std']})
renda_uf.rename(index=uf, inplace=True)
renda_uf.sort_values(('Renda', 'mean'), ascending=False)

In [ ]:
ax = sns.boxplot(x='Renda', y='UF', data=dados.query('Renda < 10000').assign(UF=lambda d: d.UF.map(uf)), orient='h')
ax.figure.set_size_inches(14, 10)
ax.set_title('Box-plot da renda por UF (até R$ 10.000)', fontsize=18);